# Agentic Workflows

We will build an agentic system that generates a short research report through planning, external tool usage, and feedback integration. Our workflow will involve:

### Agents

* **Planning Agent / Writer**: Creates an outline and coordinates tasks.
* **Research Agent**: Gathers external information using tools like Arxiv, Tavily, and Wikipedia.
* **Editor Agent**: Reflects on the report and provides suggestions for improvement.


### Research Tools

By importing `research_tools`, we gain access to several search utilities:

- `research_tools.arxiv_search_tool(query)` → search academic papers from **arXiv**  

  *Example:* `research_tools.arxiv_search_tool("neural networks for climate modeling")`

- `research_tools.tavily_search_tool(query)` → perform web searches with the **Tavily API**  

  *Example:* `research_tools.tavily_search_tool("latest trends in sunglasses fashion")`

- `research_tools.wikipedia_search_tool(query)` → retrieve summaries from **Wikipedia**  

  *Example:* `research_tools.wikipedia_search_tool("Ensemble Kalman Filter")`

Run the cell below to make them available.

In [1]:
# =========================
# Imports
# =========================

# --- Standard library 
from datetime import datetime
import re
import json
import ast


# --- Third-party ---
from IPython.display import Markdown, display
from aisuite import Client

# --- Local / project ---
import research_tools

In [2]:
import unittests

### Initialize client

Create a shared client instance for upcoming calls.

In [3]:
CLIENT = Client()

## Step 1: planner_agent

### Objective
Correctly set up a call to a language model (LLM) to generate a research plan.

### Instructions

1. **Focus Areas**:
   - Ensure `CLIENT.chat.completions.create` is correctly configured.
   - Pass the `model` and `messages` parameters correctly:
     - **Model**: Use `"openai:gpt-5-mini"` by default.
     - **Messages**: Set with `{"role": "user", "content": user_prompt}`.
     - **Temperature**: Fixed at 1 for creative outputs.

### Notes

- The prompt is pre-defined and guides the LLM on task requirements.
- Only return a formatted list of steps — no extra text.

Focus on the LLM call setup to complete the task.

In [ ]:
# GRADED FUNCTION: planner_agent

def planner_agent(topic: str, model: str = "openai:gpt-5-mini") -> list[str]:
    """
    Generates a plan as a Python list of steps (strings) for a research workflow.

    Args:
        topic (str): Research topic to investigate.
        model (str): Language model to use.

    Returns:
        List[str]: A list of executable step strings.
    """

    
    # Build the user prompt
    user_prompt = f"""
    You are a planning agent responsible for organizing a research workflow with multiple intelligent agents.

    🧠 Available agents:
    - A research agent who can search the web, Wikipedia, and arXiv.
    - A writer agent who can draft research summaries.
    - An editor agent who can reflect and revise the drafts.

    🎯 Our job is to write a clear, step-by-step research plan **as a valid Python list**, where each step is a string.
    Each step should be atomic, executable, and must rely only on the capabilities of the above agents.

    🚫 DO NOT include irrelevant tasks like "create CSV", "set up a repo", "install packages", etc.
    ✅ DO include real research-related tasks (e.g., search, summarize, draft, revise).
    ✅ DO assume tool use is available.
    ✅ DO NOT include explanation text — return ONLY the Python list.
    ✅ The final step should be to generate a Markdown document containing the complete research report.

    Topic: "{topic}"
    """

    # Add the user prompt to the messages list
    messages = [{"role": "user", "content": user_prompt}]

    ### START CODE HERE ###

    # Call the LLM
    response = CLIENT.chat.completions.create( 
        # Pass in the model
        model=model,
        # Define the messages. Remember this is meant to be a user prompt!
        messages=messages,
        # Keep responses creative
        temperature=1, 
    )

    ### END CODE HERE ###

    # Extract message from response
    steps_str = response.choices[0].message.content.strip()

    # Parse steps
    steps = ast.literal_eval(steps_str)

    return steps

In [ ]:
# Test Our code!
unittests.test_planner_agent(planner_agent)

 All tests passed!


## Step 2: research_agent

### Objective
Set up a call to a language model (LLM) to perform a research task using various tools.

### Instructions

**Focus Areas**:

- **Creating a Custom Prompt**:
  - **Define the Role**: Clearly specify the role, such as "research assistant."
  - **List Available Tools** (as strings inside the prompt, not the actual functions):
    - Use `arxiv_tool` to find academic papers.
    - Use `tavily_tool` for general web searches.
    - Use `wikipedia_tool` for accessing encyclopedic knowledge.
  - **Specify the Task**: Include a placeholder in Our prompt for defining the specific task that needs to be accomplished.
  - **Include Date Information**: Add a placeholder for the current date or time to provide context.

- **Creating Messages Dict**:
  - Ensure the `messages` are correctly set with `{"role": "user", "content": prompt}`.

- **Creating Tools List**:
  - Create a list of tools for use, such as `research_tools.arxiv_search_tool`, `research_tools.tavily_search_tool`, and `research_tools.wikipedia_search_tool`.

- **Correctly Setting the Call to the LLM**:
  - Pass the `model`, `messages`, and `tools` parameters accurately.
  - Set `tool_choice` to `"auto"` for automatic tool selection.
  - Limit interactions with `max_turns=6`.

### Notes

- The function provides pre-coded blocks where you need to replace placeholder values.
- The approach allows the LLM to use tools dynamically based on the task.

Focus on accurately setting the messages, tools, and LLM call parameters to complete the task.

In [ ]:
# GRADED FUNCTION: research_agent

def research_agent(task: str, model: str = "openai:gpt-4o", return_messages: bool = False):
    """
    Executes a research task using tools via aisuite (no manual loop).
    Returns either the assistant text, or (text, messages) if return_messages=True.
    """
    print("==================================")  
    print("🔍 Research Agent")                 
    print("==================================")

    current_time = datetime.now().strftime('%Y-%m-%d')
    
    ### START CODE HERE ###

    # Create a customizable prompt by defining the role (e.g., "research assistant"),
    # listing tools (arxiv_tool, tavily_tool, wikipedia_tool) for various searches,
    # specifying the task with a placeholder, and including a current_time placeholder.
    prompt = f"""
    You are a senior academic research assistant.
    Use the available tools to gather accurate and relevant information for the given task:
    - arxiv_tool: search academic papers on arXiv.
    - tavily_tool: search the web for general and current information.
    - wikipedia_tool: retrieve encyclopedic background knowledge from Wikipedia.

    Current date: {current_time}

    Task: {task}
    """

    # Create the messages dict to pass to the LLM. Remember this is a user prompt!
    messages = [{"role": "user", "content": prompt}]

    # Save all of Our available tools in the tools list. These can be found in the research_tools module.
    # You can identify each tool in Our list like this:
    # research_tools.<name_of_tool>, where <name_of_tool> is replaced with the function name of the tool.
    tools = [
        research_tools.arxiv_search_tool,
        research_tools.tavily_search_tool,
        research_tools.wikipedia_search_tool,
    ]

    # Call the model with tools enabled
    response = CLIENT.chat.completions.create(
        # Set the model
        model=model,
        # Pass in the messages.
        messages=messages,
        # Pass in the tools list. 
        tools=tools,
        tool_choice="auto",
        max_turns=6,
    )
    ### END CODE HERE ###

    content = response.choices[0].message.content
    print("✅ Output:\n", content)

    
    return (content, messages) if return_messages else content  

In [ ]:
# Test Our code!
unittests.test_research_agent(research_agent)

🔍 Research Agent
✅ Output:
 Here are three key references and their summaries gathered from different sources:

1. **Academic Paper: "Efficient Password-Typed Key Agreement Scheme" by Sattar J Aboud**
   - **Summary**: This research examines the verifier password-typed key agreement scheme by Lee, Kim, and Yoo, proving its insecurity. It proposes an enhanced scheme resistant to password guessing and stolen verifier attacks, promising more security and efficiency compared to its predecessor. [Read more](https://arxiv.org/pdf/1002.1163v1)

2. **Academic Paper: "Integrating HCI Datasets in Project-Based Machine Learning Courses" by Xiaodong Qu et al.**
   - **Summary**: This study explores how HCI datasets can enhance college-level machine learning courses. It identifies best practices through literature review and case studies, noting increased student engagement and addressing challenges such as data complexity. It offers recommendations to improve these educational frameworks. [Read mo

## Step 3: writer_agent

### Objective
Set up a call to a language model (LLM) for executing writing tasks like drafting, expanding, or summarizing text.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of a writing agent focused on generating academic or technical content.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The function is designed to produce well-structured text by setting the correct prompts.
- Temperature is set to 1.0 to allow for creative variance in the writing outputs.

Ensure the system prompt and messages are defined properly to achieve a structured output from the LLM.

In [8]:
# GRADED FUNCTION: writer_agent
def writer_agent(task: str, model: str = "openai:gpt-4o") -> str: # @REPLACE def writer_agent(task: str, model: str = None) -> str:
    """
    Executes writing tasks, such as drafting, expanding, or summarizing text.
    """
    print("==================================")
    print("✍️ Writer Agent")
    print("==================================")

### START CODE HERE ###
    # Create the system prompt.
    # This should assign the LLM the role of a writing agent specialized in generating well-structured academic or technical content
    system_prompt = (
        "You are an expert writing agent specialized in drafting, expanding, and "
        "summarizing clear, accurate, and well-structured academic and technical content."
    )

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role": "system", "content": system_prompt}

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role": "user", "content": task}

    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    ### END CODE HERE ###

    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=1.0
    )

    return response.choices[0].message.content

In [ ]:
# Test Our code!
unittests.test_writer_agent(writer_agent)

✍️ Writer Agent
 All tests passed!


## Step 4: editor_agent

### Objective
Configure a call to a language model (LLM) to perform editorial tasks such as reflecting, critiquing, or revising drafts.

### Instructions

1. **Focus Areas**:
   - **System Prompt**:
     - Define `system_prompt` to assign the LLM the role of an editor agent whose task is to reflect on, critique, or improve drafts.
   - **System and User Messages**:
     - Create `system_msg` using `{"role": "system", "content": system_prompt}`.
     - Create `user_msg` using `{"role": "user", "content": task}`.
   - **Messages List**:
     - Combine `system_msg` and `user_msg` into a `messages` list.

### Notes

- The editor agent is tailored for enhancing the quality of text by setting an appropriate role and task in the prompts.
- Temperature is set to 0.7, balancing creativity and coherence in editorial outputs.

Ensure the system prompt and messages are accurately set up to perform effective editorial tasks with the LLM.

In [10]:
# GRADED FUNCTION: editor_agent
def editor_agent(task: str, model: str = "openai:gpt-4o") -> str:
    """
    Executes editorial tasks such as reflection, critique, or revision.
    """
    print("==================================")
    print("🧠 Editor Agent")
    print("==================================")
    
    ### START CODE HERE ###

    system_prompt = (
        "You are a meticulous editor agent specialized in reflecting on, critiquing, "
        "and revising drafts to improve clarity, structure, accuracy, and overall quality."
    )

    # Define the system msg by using the system_prompt and assigning the role of system
    system_msg = {"role": "system", "content": system_prompt}

    # Define the user msg. In this case the user prompt should be the task passed to the function
    user_msg = {"role": "user", "content": task}

    # Add both system and user messages to the messages list
    messages = [system_msg, user_msg]
    ### END CODE HERE ###
    
    response = CLIENT.chat.completions.create(
        model=model, 
        messages=messages,
        temperature=0.7 
    )
    
    return response.choices[0].message.content

In [ ]:
# Test Our code!
unittests.test_editor_agent(editor_agent)

🧠 Editor Agent
 All tests passed!


### 🎯 The Executor Agent

The `executor_agent` manages the workflow by executing each step of a given plan. It:

1. Decides **which agent** (`research_agent`, `writer_agent`, or `editor_agent`) should handle the step.
2. Builds context from the outputs of previous steps.
3. Sends the enriched task to the selected agent.
4. Collects and stores the results in a shared history.

**Do not implement or modify this function.** It is already provided as the orchestration component of the multi-agent pipeline.

Notice that `planner_agent` might return a long list of steps. Because of this, the maximum number of steps is set to a maximum of 4 to keep running time reasonable.

In [12]:
agent_registry = {
    "research_agent": research_agent,
    "editor_agent": editor_agent,
    "writer_agent": writer_agent,
}

def clean_json_block(raw: str) -> str:
    """
    Clean the contents of a JSON block that may come wrapped with Markdown backticks.
    """
    raw = raw.strip()
    if raw.startswith("```"):
        raw = re.sub(r"^```(?:json)?\n?", "", raw)
        raw = re.sub(r"\n?```$", "", raw)
    return raw.strip()

In [ ]:
def executor_agent(topic, model: str = "openai:gpt-4o", limit_steps: bool = True):

    plan_steps = planner_agent(topic)
    max_steps = 4

    if limit_steps:
        plan_steps = plan_steps[:min(len(plan_steps), max_steps)]
    
    history = []

    print("==================================")
    print("🎯 Editor Agent")
    print("==================================")

    for i, step in enumerate(plan_steps):

        agent_decision_prompt = f"""
        You are an execution manager for a multi-agent research team.

        Given the following instruction, identify which agent should perform it and extract the clean task.

        Return only a valid JSON object with two keys:
        - "agent": one of ["research_agent", "editor_agent", "writer_agent"]
        - "task": a string with the instruction that the agent should follow

        Only respond with a valid JSON object. Do not include explanations or markdown formatting.

        Instruction: "{step}"
        """
        response = CLIENT.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": agent_decision_prompt}],
            temperature=0,
        )

        raw_content = response.choices[0].message.content
        cleaned_json = clean_json_block(raw_content)
        agent_info = json.loads(cleaned_json)

        agent_name = agent_info["agent"]
        task = agent_info["task"]

        context = "\n".join([
            f"Step {j+1} executed by {a}:\n{r}" 
            for j, (s, a, r) in enumerate(history)
        ])
        enriched_task = f"""
        You are {agent_name}.

        Here is the context of what has been done so far:
        {context}

        Our next task is:
        {task}
        """

        print(f"\n🛠️ Executing with agent: `{agent_name}` on task: {task}")

        if agent_name in agent_registry:
            output = agent_registry[agent_name](enriched_task)
            history.append((step, agent_name, output))
        else:
            output = f"⚠️ Unknown agent: {agent_name}"
            history.append((step, agent_name, output))

        print(f"✅ Output:\n{output}")

    return history

In [14]:
# If you want to see the full workflow without limiting the number of steps. Set limit_steps to False
# Keep in mind this could take more than 10 minutes to complete
executor_history = executor_agent("The ensemble Kalman filter for time series forecasting", limit_steps=True)

md = executor_history[-1][-1].strip("`")  
display(Markdown(md))

🎯 Editor Agent

🛠️ Executing with agent: `research_agent` on task: Perform a comprehensive literature search for 'ensemble Kalman filter' applied to time series forecasting across arXiv, Google Scholar, Web, and Wikipedia; return an initial candidate list of relevant papers, review articles, tutorials, and authoritative webpages with titles and URLs.
🔍 Research Agent
✅ Output:
 Here is a comprehensive list of resources related to the application of the Ensemble Kalman Filter in time series forecasting, gathered from various sources:

### Arxiv Papers
1. **Ensemble Kalman Filtering Meets Gaussian Process SSM for Non-Mean-Field and Online Inference**
   - *Authors*: Zhidi Lin, Yiyong Sun, Feng Yin, Alexandre Hoang Thiéry
   - *Published*: 2023-12-10
   - [Link to paper](http://arxiv.org/abs/2312.05910v5) | [PDF](https://arxiv.org/pdf/2312.05910v5)

2. **LLM-Mixer: Multiscale Mixing in LLMs for Time Series Forecasting**
   - *Authors*: Md Kowsher, Md. Shohanur Islam Sobuj, Nusrat Jahan Pr

Here are survey and tutorial resources along with textbook references that explain the Kalman filter fundamentals and ensemble Kalman filter theory, suitable for background study:

### Academic Papers from arXiv
1. **The Geometric Unscented Kalman Filter**
   - *Authors*: Chengling Fang, Jiang Liu, Songqing Ye, Ju Zhang
   - *Published*: 2020
   - [Read paper](http://arxiv.org/abs/2009.13079v1) | [PDF](https://arxiv.org/pdf/2009.13079v1)
   - **Summary**: Presents a new nonlinear estimation technique leveraging geometric unscented sampling strategies combining positive weights for stability.

2. **Feedback Capacity and a Variant of the Kalman Filter with ARMA Gaussian Noises**
   - *Authors*: Song Fang, Quanyan Zhu
   - *Published*: 2020
   - [Read paper](http://arxiv.org/abs/2001.03108v6) | [PDF](https://arxiv.org/pdf/2001.03108v6)
   - **Summary**: Links a feedback channel to Kalman filtering, providing explicit bounds for feedback capacity with ARMA Gaussian noises.

3. **A summary on the UD Kalman Filter**
   - *Authors*: J. Humberto Ramos, Kevin Brink, Prashant Ganesh, John E. Hurtado
   - *Published*: 2022
   - [Read paper](http://arxiv.org/abs/2203.06105v2) | [PDF](https://arxiv.org/pdf/2203.06105v2)
   - **Summary**: Provides a concise reference on the UD Kalman filter, including matrix algorithms and references.

### Web Articles and Tutorials
1. **Understanding the Ensemble Kalman Filter**
   - [Read tutorial](https://www.math.umd.edu/~slud/RITF17/enkf-tutorial.pdf)
   - **Content**: Explains the ensemble Kalman filter (EnKF) with core steps, focusing on forecast and update mechanisms in stochastic environments.

2. **The Ensemble Kalman Filter: Theoretical Formulation and Practical Implementation**
   - [Read document](https://www.ecmwf.int/sites/default/files/elibrary/2003/9321-ensemble-kalman-filter-theoretical-formulation-and-practical-implementation.pdf)
   - **Content**: Provides a three-stage presentation of EnKF, representing error statistics, prediction, and analysis schemes.

3. **GitHub - Kalman and Bayesian Filters in Python**
   - [Visit repository](https://github.com/rlabbe/Kalman-and-Bayesian-Filters-in-Python)
   - **Content**: A practical introduction to Kalman filters, including exercises and solutions in Python, with a focus on understanding and implementing various filter types.

### Textbooks and Reference Books
1. **Data Assimilation: The Ensemble Kalman Filter**
   - [Springer Link](https://link.springer.com/book/10.1007/978-3-642-03711-5)
   - **Overview**: Covers data assimilation with Kalman filters, emphasizing mathematical frameworks suitable for integrating dynamics with measurements.

2. **The Extended and Ensemble Kalman Filters - Inverse Problems and Data Assimilation**
   - [Cambridge Core](https://www.cambridge.org/core/books/inverse-problems-and-data-assimilation/extended-and-ensemble-kalman-filters/C8BF020CD2525369E8EC5F04D28E4FD7)
   - **Content**: Describes both ExKF and EnKF, focusing on linearization and empirical covariance.

### Wikipedia
- [Kalman filter - Wikipedia](https://en.wikipedia.org/wiki/Kalman_filter)
  - **Summary**: Kalman filtering is an algorithm used to refine state estimates based on measurements over time, minimizing mean squared error.

These resources provide thorough explanations and explorations of both traditional and ensemble Kalman filtering, suitable for students and researchers seeking a deeper understanding of the topic.